# **LangChain 이해와 활용1**



## **1.환경준비**

### (1) 구글 드라이브

#### 1) 구글 드라이브 폴더 생성
* 새 폴더(langchain)를 생성하고
* 제공 받은 파일을 업로드

#### 2) 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### (2) 라이브러리

#### 1) 필요한 라이브러리 설치

In [ ]:
!pip install langchain langchain-openai langchain_community -q

#### 2) 라이브러리 로딩

In [3]:
import pandas as pd
import numpy as np
import os
import openai

from langchain.prompts import PromptTemplate
from langchain.prompts.chat import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from langchain.schema import HumanMessage, SystemMessage, AIMessage

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

c:\Users\USER\anaconda3\envs\aivle_pytorch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### (3) OpenAI API Key 확인

In [5]:
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

path = '../'

# API 키 로드 및 환경변수 설정
load_api_keys(path + 'API_KEY.env')

* ⚠️ 아래 코드셀은, 실행해서 key가 제대로 보이는지 확인하고 삭제하세요.

In [6]:
print(os.environ['OPENAI_API_KEY'][:40])

sk-proj-NZCTfKEaIvmWyBt61dmY6ANFdYXgg-GU


## **2. Model**

### (1) 모델 선언

In [7]:
chat = ChatOpenAI(model_name="gpt-4.1-mini", verbose=True)

### (2) 사용하기1

In [8]:
chat.invoke("세계에서 가장 큰 산은?")

AIMessage(content='세계에서 가장 큰 산은 에베레스트 산(Mount Everest)입니다. 해발 고도는 약 8,848.86미터로, 지구상에서 가장 높은 산으로 알려져 있습니다. 에베레스트는 네팔과 중국(티베트 자치구)의 국경에 위치해 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 14, 'total_tokens': 83, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_4c2851f862', 'id': 'chatcmpl-CTece6uxiVQWOxaODFFWif3jSimnM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--f140d2ca-de46-47ae-a43b-8cceae49ff2d-0', usage_metadata={'input_tokens': 14, 'output_tokens': 69, 'total_tokens': 83, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### (3) 사용하기2 : 역할 부여

In [9]:
# 역할부여
sys_role = '당신은 애국심을 가지고 있는 건전한 대한민국 국민입니다.'
question = "독도는 어느나라 땅이야?"

result = chat.invoke([HumanMessage(content = question), SystemMessage(content = sys_role)])
result

AIMessage(content='독도는 대한민국의 영토입니다. 역사적, 지리적, 국제법적으로 명확히 대한민국의 땅임을 인정받고 있으며, 대한민국 정부가 실효적으로 지배하고 있습니다. 독도의 주권을 지키는 것은 우리 국민 모두의 중요한 책임입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 37, 'total_tokens': 98, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_4c2851f862', 'id': 'chatcmpl-CTeclMjfvI7KvMkI72iznJroA2KQ2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--ff9dcb01-8575-4d8d-90f5-498ff352f959-0', usage_metadata={'input_tokens': 37, 'output_tokens': 61, 'total_tokens': 98, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## **3.ChatPromptTemplate**

- 시스템 메시지, 사용자 메시지, AI 메시지 등 역할(role) 구분
- 다중 메시지 기반의 프롬프트 흐름을 구성할 수 있도록 도와주는 템플릿

**메세지 종류**
- SystemMessage : AI에게 역할/성격을 지정
- HumanMessage : 사용자 질문 또는 요청
- AIMessage : AI 응답


여러 메시지를 구조화하여 대화 설계

### (1) 역할 부여

In [10]:
s_msg = "너는 친절하고 유머 있는 상담사야."
h_msg = "요즘 너무 지치고 의욕이 없어. 어떻게 하면 좋을까?"

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", s_msg),
    ("human", h_msg),
])

llm = ChatOpenAI(model_name = 'gpt-4.1-mini',
                 temperature=1, top_p = 0.95)  # 답변의 다양성 제어

messages = chat_prompt.format_messages() # 실제 메시지 객체 리스트를 생성
response = llm.invoke(messages)
print(response.content)

아이고, 지친 마음에 의욕까지 달아나버리면 정말 힘들죠. 잠깐만요, 제가 마음 회복 비법을 알려드릴게요!

1. **작은 성공 맛보기**: 큰 일은 잠시 제쳐두고, ‘양치 3초 더 하기’나 ‘커피 한 잔 제대로 즐기기’ 같은 아주 작은 목표부터 시작해 보세요. 의욕은 작은 성공에서 자라납니다!

2. **몸 움직이기**: 몸이 움직이면 뇌도 깨어난다고 하잖아요. 가볍게 산책하거나 스트레칭만 해도 기분이 한결 나아질 거예요.

3. **나에게 관대해지기**: ‘해야 한다’ 목록에서 ‘해도 된다’ 목록으로 마인드를 바꿔 보세요. 오늘은 좀 쉬어도 괜찮아요, 휴식도 중요한 일입니다!

4. **웃음 코드 켜기**: 유머나 웃음은 최고의 에너지 충전제! 좋아하는 코미디 영상 한 편, 웃음 만랩 준비해 보시는 건 어떨까요?

어때요, 같이 천천히 다시 힘 내 볼까요? 필요하면 언제든 얘기해 주세요!


In [11]:
chat_prompt

ChatPromptTemplate(input_variables=[], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='너는 친절하고 유머 있는 상담사야.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='요즘 너무 지치고 의욕이 없어. 어떻게 하면 좋을까?'), additional_kwargs={})])

In [12]:
messages

[SystemMessage(content='너는 친절하고 유머 있는 상담사야.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='요즘 너무 지치고 의욕이 없어. 어떻게 하면 좋을까?', additional_kwargs={}, response_metadata={})]

In [13]:
response

AIMessage(content='아이고, 지친 마음에 의욕까지 달아나버리면 정말 힘들죠. 잠깐만요, 제가 마음 회복 비법을 알려드릴게요!\n\n1. **작은 성공 맛보기**: 큰 일은 잠시 제쳐두고, ‘양치 3초 더 하기’나 ‘커피 한 잔 제대로 즐기기’ 같은 아주 작은 목표부터 시작해 보세요. 의욕은 작은 성공에서 자라납니다!\n\n2. **몸 움직이기**: 몸이 움직이면 뇌도 깨어난다고 하잖아요. 가볍게 산책하거나 스트레칭만 해도 기분이 한결 나아질 거예요.\n\n3. **나에게 관대해지기**: ‘해야 한다’ 목록에서 ‘해도 된다’ 목록으로 마인드를 바꿔 보세요. 오늘은 좀 쉬어도 괜찮아요, 휴식도 중요한 일입니다!\n\n4. **웃음 코드 켜기**: 유머나 웃음은 최고의 에너지 충전제! 좋아하는 코미디 영상 한 편, 웃음 만랩 준비해 보시는 건 어떨까요?\n\n어때요, 같이 천천히 다시 힘 내 볼까요? 필요하면 언제든 얘기해 주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 275, 'prompt_tokens': 40, 'total_tokens': 315, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_4c2851f862', 'id': 'chatcmpl-CTecqeMiHUCDDq349aYYFaTbAIdZt', 'service_tier': 'default', 'finish_reason': 'st

In [14]:
print(response.content)

아이고, 지친 마음에 의욕까지 달아나버리면 정말 힘들죠. 잠깐만요, 제가 마음 회복 비법을 알려드릴게요!

1. **작은 성공 맛보기**: 큰 일은 잠시 제쳐두고, ‘양치 3초 더 하기’나 ‘커피 한 잔 제대로 즐기기’ 같은 아주 작은 목표부터 시작해 보세요. 의욕은 작은 성공에서 자라납니다!

2. **몸 움직이기**: 몸이 움직이면 뇌도 깨어난다고 하잖아요. 가볍게 산책하거나 스트레칭만 해도 기분이 한결 나아질 거예요.

3. **나에게 관대해지기**: ‘해야 한다’ 목록에서 ‘해도 된다’ 목록으로 마인드를 바꿔 보세요. 오늘은 좀 쉬어도 괜찮아요, 휴식도 중요한 일입니다!

4. **웃음 코드 켜기**: 유머나 웃음은 최고의 에너지 충전제! 좋아하는 코미디 영상 한 편, 웃음 만랩 준비해 보시는 건 어떨까요?

어때요, 같이 천천히 다시 힘 내 볼까요? 필요하면 언제든 얘기해 주세요!


### (2) 메시지 흐름 관리

In [15]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 요리 전문가야."),
    ("human", "간단한 점심 추천해줘."),
    ("ai", "김치볶음밥은 어때? 간단하고 맛있어."),
    ("human", "좋아! 다른 메뉴도 하나만 더 알려줘.")
])

In [16]:
messages = chat_prompt.format_messages()
messages

[SystemMessage(content='너는 요리 전문가야.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='간단한 점심 추천해줘.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='김치볶음밥은 어때? 간단하고 맛있어.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='좋아! 다른 메뉴도 하나만 더 알려줘.', additional_kwargs={}, response_metadata={})]

In [17]:
response = llm.invoke(messages)
print(response.content)

계란 토스트 추천할게! 식빵에 계란을 입혀서 구우면 간단하고 든든한 한 끼가 돼. 케첩이나 마요네즈를 곁들이면 더 맛있어.


### (3) 입력변수 사용

In [18]:
s_msg = "너는 {role}야."
h_msg = "요즘 너무 지치고 의욕이 없어. 어떻게 하면 좋을까?"

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", s_msg),
    ("human", h_msg),
])

print(chat_prompt)

input_variables=['role'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['role'], input_types={}, partial_variables={}, template='너는 {role}야.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='요즘 너무 지치고 의욕이 없어. 어떻게 하면 좋을까?'), additional_kwargs={})]


In [19]:
messages = chat_prompt.format_messages(role="친절한 상담사")
print(messages)

[SystemMessage(content='너는 친절한 상담사야.', additional_kwargs={}, response_metadata={}), HumanMessage(content='요즘 너무 지치고 의욕이 없어. 어떻게 하면 좋을까?', additional_kwargs={}, response_metadata={})]


### (4) 😀실습

* 영화 추천 템플릿 만들기
    * 입력변수 : 장르
    * 장르를 입력받아 영화 1편과 추천이유를 설명하는 템플릿을 만들고 사용해 봅시다.

* 프롬프트 템플릿

In [ ]:
# 1. 프롬프트 템플릿 생성


In [ ]:
# 2. 변수 넣어서 메시지 포맷


# 3. 모델 생성 및 응답 받기


# 4. 결과 출력


## **4. OutputParser**

**Output Parser**
- LLM에서 반환된 자유형 텍스트(string)를 우리가 원하는 형태로 가공해주는 도구

**Output Parser의 종류**
- CommaSeparatedListOutputParser : 쉼표 구분 문자열 → 리스트로 변환
- PydanticOutputParser : 텍스트 → Pydantic 모델로 파싱
- StructuredOutputParser : JSON 기반 구조화 파싱

### (1) PydanticOutputParser

#### 1) Pydantic

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

#### 2) 출력파서로 이용

In [ ]:
from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser

In [ ]:
# 1. Pydantic 모델 정의
class BookInfo(BaseModel):
    title: str
    author: str
    year: int

# 2. 파서 생성
parser = PydanticOutputParser(pydantic_object=BookInfo)

# 3. 프롬프트 구성 (ChatPromptTemplate 사용)
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 책 추천 전문가야."),
    ("human", "좋은 책 하나만 추천해줘. 제목과 저자, 출판년도를 알려줘."),
    ("system", "{format_instructions}")  # 파서가 제공한 응답 형식 가이드
])

# 4. 메시지 생성
messages = prompt.format_messages(
    format_instructions=parser.get_format_instructions()
)

# 5. LLM 호출 및 파싱
llm = ChatOpenAI(model_name = 'gpt-4.1-mini', temperature=0.5, model_kwargs={"top_p": 0.95})
response = llm.invoke(messages)
book = parser.parse(response.content)

# 6. 결과 출력
print(book)

In [ ]:
messages

### (2) 😀실습

* 장르를 입력하면, 영화 제목, 감독, 주연배우, 연도를 출력하도록 합시다. (PydanticOutputParser사용)

In [ ]:
# 1. Pydantic 모델 정의


# 2. 파서 생성


# 3. 프롬프트 구성


# 4. 메시지 구성


# 5. 모델 호출 및 파싱


# 6. 결과 리스트로 변환


### (3) [참조] CommaSeparatedListOutputParser

In [ ]:
from langchain.output_parsers import CommaSeparatedListOutputParser

In [ ]:
# 출력 파서 선언
parser = CommaSeparatedListOutputParser()

# 입력 프롬프트 구성
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 사용자 취향을 정리해주는 전문가야."),
    ("human", "10대 남학생들이 좋아하는 음식 5가지를 콤마로 구분해서 말해줘."),
    ("system", "{format_instructions}")
])

# 프롬프트 구성
formatted_messages = prompt.format_messages(
    format_instructions=parser.get_format_instructions()
)

llm = ChatOpenAI(model_name = 'gpt-4.1-mini', temperature=0.5,  top_p = 0.95)
response = llm.invoke(formatted_messages)
print(parser.parse(response.content))

In [ ]:
formatted_messages

### (4) [참조] StructuredOutputParser

예제: 뉴스 요약 추출 (주석 포함)

In [ ]:
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

# 1. 출력 스키마 정의 (뉴스 제목, 요약)
schemas = [
    ResponseSchema(name="headline", description="뉴스 제목"),
    ResponseSchema(name="summary", description="뉴스 내용을 한 문장으로 요약")
]

# 2. 파서 생성
parser = StructuredOutputParser.from_response_schemas(schemas)

# 3. 프롬프트 구성 (메시지 기반)
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 뉴스 요약 도우미야."),
    ("human", "최근 흥미로운 스포츠 뉴스를 하나 소개하고 제목과 요약을 알려줘."),
    ("system", "{format_instructions}")  # 여기서 파서가 제공한 형식 안내문이 들어감
])

# 4. 실제 메시지 포맷팅
messages = prompt.format_messages(
    format_instructions=parser.get_format_instructions()
)

# 5. 모델 호출
llm = ChatOpenAI(model_name = 'gpt-4.1-mini', temperature=0.5,  top_p = 0.95)
response = llm.invoke(messages)

# 6. 결과 파싱 (JSON → dict)
result = parser.parse(response.content)
print(result)

In [ ]:
schemas

In [ ]:
messages

### [추가] prompt & output

* ChatPromptTemplate.from_template 이용
* 프롬프트 템플릿 안에 출력 구조를 제시
* 문자열로 된 딕셔너리 형태의 값을 진짜 딕셔너리로 변환하기

In [ ]:
from langchain.prompts.chat import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI

# LLM 객체 설정 (예시: OpenAI GPT)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

In [ ]:
# 프롬프트 템플릿 정의
template = ChatPromptTemplate.from_template("""
당신은 {role}입니다.
다음의 정보를 참조해서 답변해주세요.
[참고 정보]
- 정보1 : {info1}
- 정보2 : {info2}
질문 : AI 기술을 거대기업이 주도하는데에 따른 문제점과 해결책은?

출력은 다음의 구조에 맞춰 주세요.
출력 예시
{{
    '문제점':'문제점을 여기에 적어주세요.',
    '해결책':'해결책을 여기에 적어주세요.'
}}
""")

In [ ]:
# 사용할 변수들 정의
role = "기술 정책 전문가"
info1 = "AI 기술 개발은 높은 자본과 인프라가 필요하여 대기업이 시장을 선점하고 있음"
info2 = "중소기업과 개인 개발자의 접근성은 점점 낮아지고 있음"

# 프롬프트 구성( .format)
formatted_prompt = template.format(role = role, info1 = info1, info2 = info2)

# LLM 호출
response = llm.invoke(formatted_prompt)
result = response.content.strip()

# 출력
print(result)

In [ ]:
# result의 type은?
type(result)

In [ ]:
# 문자열을 진짜 딕셔너리로 변환하기
import ast
result_dict = ast.literal_eval(result)

type(result_dict)